# Medallion Architecture

**Bronze** --> Preserve raw/source data => raw ingestion \
**Silver** --> validated/cleaned/deduplicated data => casts, null handling, deduplication, joins \
**Gold** --> business-ready aggregation/modeling => metrics, dimensions, aggregates

###  Create schemas for the three layers


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.customer360_bronze;
CREATE SCHEMA IF NOT EXISTS workspace.customer360_silver;
CREATE SCHEMA IF NOT EXISTS workspace.customer360_gold;

### BRONZE customer Data


In [0]:
from pyspark.sql.functions import *

custRaw_data = [
    (1001, " john smith ", "ACTIVE", "Dallas", "TX", "214-555-1001"),
    (1002, "MARY JONES", "Active", "Plano", "tx", "(469) 555-1002"),
    (1003, "Steve Brown", "active", "Frisco", "TX", "972.555.1003"),
    (1003, "Steve Brown", "active", "Frisco", "TX", "972.555.1003"),
    (1004, "Sarah Wilson", None, "Irving", "TX", None)

]

custRaw_columns = [ "custId", "custName","status", "city", "state", "phone" ]

dfCustomerRaw = spark.createDataFrame(custRaw_data, custRaw_columns)

#add ingestion metadata
dfCustomerBronze = ( 
                    dfCustomerRaw
                    .withColumn("ingestionTimestamp", current_timestamp())
                    .withColumn("sourceSystem", lit("CRM"))
)

dfCustomerBronze.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_bronze.customer_raw")


In [0]:
%sql
select * from customer360_bronze.customer_raw;

# **Create BRONZE- Account table**


In [0]:
accountRaw_data = [
    (5001, 1001, "Wireless", "Active", 120.50),
    (5002, 1001, "Internet", "Active", 75.00),
    (5003, 1002, "Fiber", "Active", 145.99),
    (5004, 1003, "Wireless", "Suspended", 85.25),
    (5005, 9999, "Fiber", "Active", 110.00)
]

accountRaw_columns = [
    "acctId",
    "custId",
    "serviceType",
    "accountStatus",
    "balance"
]

dfAccountRaw = spark.createDataFrame( accountRaw_data, accountRaw_columns)

dfAccountBronze = (
    dfAccountRaw
    .withColumn("ingestionTimestamp", current_timestamp())
    .withColumn("sourceSystem", lit("Billing"))
)

dfAccountBronze.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_bronze.account_raw" )

In [0]:
%sql
select * from customer360_bronze.account_raw;

In [0]:
dfBronzeCustomers = spark.table(
    "workspace.customer360_bronze.customer_raw"
)

### SILVER REQUIREMENT
Clean Customer:

custName → trim + proper case \
status → proper/consistent case\
city → proper case\
state → uppercase\
phone → digits only\
remove duplicate customers\

In [0]:
from pyspark.sql.functions import *

#Clean custName
dfCleanCustName= dfBronzeCustomers.withColumn("custName", trim(initcap(col("custName"))))
#display(dfCleanCustName)

#clean status
dfCleanStatus = dfCleanCustName.withColumn("status", trim(initcap(col("status"))))
#display(dfCleanStatus)

#clean location (city/state)
dfCleanLocation = (
    dfCleanStatus
    .withColumn("city", trim(initcap(col("city"))))
    .withColumn("state", trim(upper(col("state"))))
)
#display(dfCleanLocation)

dfCleanPh = dfCleanLocation.withColumn("phone",regexp_replace(col("phone"), "[^0-9]", ""))
#display(dfCleanPh)

dfDeDupRecords = dfCleanPh.dropDuplicates(["custId"])
#display(dfDeDupRecords)
dfDeDupRecords.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_silver.customer_clean")




In [0]:
%sql
select * from  workspace.customer360_silver.customer_clean;

Clean Account:
- Standardize serviceType 
- Standardize accountStatus 
- Make sure balance has appropriate numeric datatype 
- Identify orphan accounts 

In [0]:
%sql
select * from customer360_bronze.account_raw

In [0]:
%python
dfSilverCustClean = spark.table(  "workspace.customer360_silver.customer_clean")
dfSilverAcctClean = spark.table(  "workspace.customer360_bronze.account_raw")

c = dfSilverCustClean.alias("c")
a = dfSilverAcctClean.alias("a")

dfOrphanAcctounts = a.join(
    c,
    c.custId == a.custId,
    how="left_anti"
)

#display(dfOrphanAcctounts)
dfOrphanAcctounts.write.format("delta").mode("overwrite").saveAsTable("workspace.customer360_silver.orphan_accounts")

In [0]:
%sql
select * from workspace.customer360_silver.orphan_accounts;

In [0]:
%python
from pyspark.sql.functions import *

dfSilverCustClean1 = spark.table(  "workspace.customer360_silver.customer_clean")
dfSilverAcctClean1 = spark.table(  "workspace.customer360_bronze.account_raw")

# Load tables and create aliases
c = dfSilverCustClean1.alias("c")
a = dfSilverAcctClean1.alias("a")

# Perform full outer join
dfValidAccounts= c.join(
    a,
    c.custId == a.custId,
    how="inner"
).drop(a.custId, a.ingestionTimestamp, a.sourceSystem)

dfValidAccounts.write.mode("overwrite").saveAsTable("customer360_silver.accounts")

#display(dfValidAccounts)

# **Build Customer360 Silver**

### Combine--> Silver Customers + Silver accounts

In [0]:
%sql
select * from customer360_silver.customer_clean;

In [0]:
%sql
select * from customer360_silver.accounts;

In [0]:
%python
from pyspark.sql.functions import *

dfSilverCust = spark.table(  "workspace.customer360_silver.customer_clean")
dfSilverAcct = spark.table(  "workspace.customer360_silver.accounts")

# Load tables and create aliases
c = dfSilverCust.alias("c")
a = dfSilverAcct.alias("a")

# Perform full outer join
dfCustomerAccountSilver= c.join(
    a,
    c.custId == a.custId,
    how="left"
)

dfCustomerAccountSilver= dfCustomerAccountSilver.select(c.custId, c.custName, c.status, c.city, c.state, c.phone, a.acctId, a.serviceType, a.accountStatus,a.balance).orderBy("custId")
dfCustomerAccountSilver.write.mode("overwrite").saveAsTable("customer360_silver.customer_accounts")


In [0]:
%sql
select * from customer360_silver.customer_accounts;

In [0]:
%python

from pyspark.sql.functions import *

dfSilverCust = spark.table(  "workspace.customer360_silver.customer_clean")
dfSilverAcct = spark.table(  "workspace.customer360_silver.accounts")

c = dfSilverCust.alias("c")
a = dfSilverAcct.alias("a")

dfCustomerGold = c.join(
    a,
    c.custId == a.custId,
    how="left"
)

dfFinalCustomerGold = (
    dfCustomerGold.groupBy(
        c.custId
        , c.custName
        , c.city
        , c.status
    ).agg(
        countDistinct(a.acctId).alias("TotalAccountCount")
        , count(when(a.accountStatus == "Active", a.custId)).alias("ActiveAccountCount")
        # , .agg(count(lit(1)).alias("TotalAcctPerCust")
        , coalesce(avg(a.balance), lit(0)).alias("AvgBalancePerCust")
        , coalesce(sum(a.balance), lit(0)).alias("TotalBalancePerCust")
    )
    .withColumnRenamed("status", "CustomerStatus")
    .orderBy("custId")
)

dfFinalCustomerGold.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("customer360_gold.customer_summary")

In [0]:
%sql
select * from customer360_gold.customer_summary;